# Archive Suite — Google Colab GPU Pack Runner (Google AI Ultra)

Runs any staged Archive Suite benchmark pack from `My Drive/ArchiveSuite-Colab/packs/<PACK_NAME>/` against the frozen `public895` (895 pages) and `truth99` (99 owner pages) datasets, writing resume-safe per-page outputs directly to `My Drive/ArchiveSuite-Colab/results/<PACK_NAME>/`.

**Recommended runtime (Google AI Ultra):**
- **A100 80 GB / H100 80 GB + High-RAM** for 27B FP8 models or fast full sweeps.
- **L4 24 GB + High-RAM** for 1B–9B OCR models.
- Enable **Background execution** (Colab Pro+ / AI Ultra) so runs continue if your laptop sleeps.

In [ ]:
# 1. Mount Google Drive and inspect GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ArchiveSuite-Colab')
PACK_NAME = ''  # <-- Set to the folder name inside ArchiveSuite-Colab/packs/
VLLM_VERSION = '0.31.0'  # Pinned to match Sherlock's vllm/vllm-openai:v0.31.0 container
INSTALL_MINERU = False   # Set True only if the pack includes mineru25-pro

# Keep Hugging Face weights & caches on fast local Colab NVMe (never slow Drive)
os.environ['HF_HOME'] = '/content/hf'
os.environ['XDG_CACHE_HOME'] = '/content/cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
assert DRIVE_ROOT.is_dir(), f'Missing {DRIVE_ROOT} — upload ArchiveSuite-Colab to My Drive first.'
print('Available packs:', sorted(p.name for p in (DRIVE_ROOT / 'packs').iterdir() if p.is_dir()))

In [ ]:
# 2. Install pinned vLLM (+ optional mineru-vl-utils)
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'vllm=={VLLM_VERSION}', 'huggingface_hub', 'pillow'])
if INSTALL_MINERU:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mineru-vl-utils'])
import torch, vllm
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'vllm:', vllm.__version__)

In [ ]:
# 3. Run the selected pack (resume-safe: completed pages are never re-read)
import json, shutil, subprocess, sys, os
from pathlib import Path

assert PACK_NAME, 'Set PACK_NAME in Cell 1 before running.'
PACK_DIR = DRIVE_ROOT / 'packs' / PACK_NAME
RES_DIR = DRIVE_ROOT / 'results' / PACK_NAME
RES_DIR.mkdir(parents=True, exist_ok=True)

# Copy support runners to local /content/runners for fast imports
RUNNERS = Path('/content/runners')
if RUNNERS.exists():
    shutil.rmtree(RUNNERS)
if (PACK_DIR / 'colab_support').is_dir():
    shutil.copytree(PACK_DIR / 'colab_support', RUNNERS)
else:
    RUNNERS.mkdir(parents=True)

CANDIDATES = PACK_DIR / 'candidates.json'
if CANDIDATES.is_file():
    # OCR family-coverage style pack (public895 + truth99)
    cands = json.loads(CANDIDATES.read_text())
    for entry in cands:
        key = entry['key']
        print(f'=== Running {key} ({entry.get("model")}) ===')
        for split, ds_dir, manifest in [
            ('895', DRIVE_ROOT / 'datasets' / 'public895', 'public895.jsonl'),
            ('96', DRIVE_ROOT / 'datasets' / 'truth99', 'truth99.jsonl'),
        ]:
            if split not in entry.get('sets', ['895', '96']):
                continue
            sess = RES_DIR / split / key
            sess.mkdir(parents=True, exist_ok=True)
            # Prepare snapshot & prompt on local /content/hf, writing prepared.json into sess
            env = os.environ.copy()
            env['SESSION_ROOT'] = str(sess)
            env['PYTHONPATH'] = str(RUNNERS) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
            subprocess.check_call([sys.executable, str(RUNNERS / 'family_setup.py'), 'prepare', str(sess), str(CANDIDATES), key], env=env)
            # Symlink manifests into sess.parent so family_run finds pilot3.jsonl + manifest
            for mf in ('pilot3.jsonl', manifest):
                link = sess.parent / mf
                if not link.exists():
                    shutil.copy2(ds_dir / mf, link)
            subprocess.check_call([sys.executable, str(RUNNERS / 'family_run.py'), manifest], env=env)
    print('ALL CANDIDATES FINISHED ->', RES_DIR)
else:
    print('No candidates.json found in', PACK_DIR, '— inspect pack contents:')
    for p in sorted(PACK_DIR.iterdir()):
        print(' ', p.name)

In [ ]:
# 4. Completion summary
import json
from pathlib import Path

RES_DIR = DRIVE_ROOT / 'results' / PACK_NAME
for status_file in sorted(RES_DIR.rglob('status.json')):
    rel = status_file.relative_to(RES_DIR)
    st = json.loads(status_file.read_text())
    print(rel, '->', {k: st.get(k) for k in ('state', 'done', 'total', 'errors', 'capped') if k in st})